<a href="https://colab.research.google.com/github/0301sdh/faceguard/blob/main/notebooks/simswap_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is a simple example of SimSwap on processing video with multiple faces. You can change the codes for inference based on our other scripts for image or single face swapping.

Code path: https://github.com/neuralchen/SimSwap

Paper path: https://arxiv.org/pdf/2106.06340v1.pdf or https://dl.acm.org/doi/10.1145/3394171.3413630

In [2]:
## make sure you are using a runtime with GPU
## you can check at Runtime/Change runtime type in the top bar.
!nvidia-smi

Sun Oct  4 06:37:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Installation

All file changes made by this notebook are temporary.
You can try to mount your own google drive to store files if you want.


In [3]:
!git clone https://github.com/neuralchen/SimSwap
!cd SimSwap && git pull

Cloning into 'SimSwap'...
remote: Enumerating objects: 1141, done.
remote: Counting objects: 100% (508/508), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 1141 (delta 419), reused 382 (delta 382), pack-reused 633 (from 1)
Receiving objects: 100% (1141/1141), 211.46 MiB | 37.95 MiB/s, done.
Resolving deltas: 100% (601/601), done.
Already up to date.


In [4]:
!pip install insightface==0.2.1 onnxruntime moviepy
!pip install googledrivedownloader
!pip install imageio==2.4.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 111.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 32.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 50.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for imageio: filename=imageio-2.4.1-py3-none-any.whl size=3303998 sha256=b113d3428d0f648ed1374358f164ef46aac40df3d5d24783a940845203cd875f
  Stored in directory: /root/.cache/pip/wheels/dc/f7/20/a76aa96b8c2d3212ff4633d433a08cbbdd345a401f4ed7af10
Successfully built imageio
  Attempting uninstall: imageio
    Found existing installation: ImageIO 2.37.4
    Uninstalling ImageIO-2.37.4:
      Successfully uninstalled ImageIO-2.37.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-image 0.25.2 requires imageio!=2.35.0,>=2.33, but you have imageio 2.4.1 which is incompatible.
moviepy 1.0.3 requires imageio<3.0,>=2.5; python_version >= "3.4", but you have imageio 2.4.1 which is incompatible.


In [1]:
import os
os.chdir("SimSwap")
!ls

 cog.yaml	       README.md
 crop_224	      'SimSwap colab.ipynb'
 data		       simswaplogo
 demo_file	       test_one_image.py
 docs		       test_video_swapmulti.py
 download-weights.sh   test_video_swap_multispecific.py
 insightface_func      test_video_swapsingle.py
 LICENSE	       test_video_swapspecific.py
 models		       test_wholeimage_swapmulti.py
 MultiSpecific.ipynb   test_wholeimage_swap_multispecific.py
 options	       test_wholeimage_swapsingle.py
 output		       test_wholeimage_swapspecific.py
 parsing_model	       train.ipynb
 pg_modules	       train.py
 predict.py	       util


In [2]:
#from google_drive_downloader import GoogleDriveDownloader

### it seems that google drive link may not be permenant, you can find this ID from our open url.
# GoogleDriveDownloader.download_file_from_google_drive(file_id='1TLNdIufzwesDbyr_nVTR7Zrx9oRHLM_N',
#                                     dest_path='./arcface_model/arcface_checkpoint.tar')
# GoogleDriveDownloader.download_file_from_google_drive(file_id='1PXkRiBUYbu1xWpQyDEJvGKeqqUFthJcI',
#                                     dest_path='./checkpoints.zip')

!wget -P ./arcface_model https://github.com/neuralchen/SimSwap/releases/download/1.0/arcface_checkpoint.tar
!wget https://github.com/neuralchen/SimSwap/releases/download/1.0/checkpoints.zip
!unzip ./checkpoints.zip  -d ./checkpoints
!wget -P ./parsing_model/checkpoint https://github.com/neuralchen/SimSwap/releases/download/1.0/79999_iter.pth

--2026-10-04 06:38:11--  https://github.com/neuralchen/SimSwap/releases/download/1.0/arcface_checkpoint.tar
Resolving github.com (github.com)... 20.29.134.23
Connecting to github.com (github.com)|20.29.134.23|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/374891081/f01468b3-446b-4867-8c78-6d496183f9e6?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-10-04T07%3A18%3A01Z&rscd=attachment%3B+filename%3Darcface_checkpoint.tar&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-10-04T06%3A17%3A48Z&ske=2026-10-04T07%3A18%3A01Z&sks=b&skv=2018-11-09&sig=Q0pldABvysdGjZXdj%2F3siyWGGWLRZmfyJvs8uj0qfTk%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc5MTA5OTQ5MSwibmJmIjoxNzkxMDk1ODkxLCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZH

In [3]:
## You can upload filed manually
# from google.colab import drive
# drive.mount('/content/gdrive')

### Now onedrive file can be downloaded in Colab directly!
### If the link blow is not permanent, you can just download it from the
### open url(can be found at [our repo]/doc/guidance/preparation.md) and copy the assigned download link here.
### many thanks to woctezuma for this very useful help
!wget --no-check-certificate "https://sh23tw.dm.files.1drv.com/y4mmGiIkNVigkSwOKDcV3nwMJulRGhbtHdkheehR5TArc52UjudUYNXAEvKCii2O5LAmzGCGK6IfleocxuDeoKxDZkNzDRSt4ZUlEt8GlSOpCXAFEkBwaZimtWGDRbpIGpb_pz9Nq5jATBQpezBS6G_UtspWTkgrXHHxhviV2nWy8APPx134zOZrUIbkSF6xnsqzs3uZ_SEX_m9Rey0ykpx9w" -O antelope.zip
!unzip ./antelope.zip -d ./insightface_func/models/


--2026-10-04 06:38:27--  https://sh23tw.dm.files.1drv.com/y4mmGiIkNVigkSwOKDcV3nwMJulRGhbtHdkheehR5TArc52UjudUYNXAEvKCii2O5LAmzGCGK6IfleocxuDeoKxDZkNzDRSt4ZUlEt8GlSOpCXAFEkBwaZimtWGDRbpIGpb_pz9Nq5jATBQpezBS6G_UtspWTkgrXHHxhviV2nWy8APPx134zOZrUIbkSF6xnsqzs3uZ_SEX_m9Rey0ykpx9w
Resolving sh23tw.dm.files.1drv.com (sh23tw.dm.files.1drv.com)... 150.171.22.11, 2620:1ec:50::11
Connecting to sh23tw.dm.files.1drv.com (sh23tw.dm.files.1drv.com)|150.171.22.11|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-10-04 06:38:27 ERROR 404: Not Found.

Archive:  ./antelope.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of ./antelope.zip or
        ./antelope.zip.zip, and cannot find ./antelope.zip.ZIP, period.


## Inference

In [4]:
import cv2
import torch
import fractions
import numpy as np
from PIL import Image
import torch.nn.functional as F
from torchvision import transforms
from models.models import create_model
from options.test_options import TestOptions
from insightface_func.face_detect_crop_multi import Face_detect_crop
from util.videoswap import video_swap
from util.add_watermark import watermark_image

/usr/local/lib/python3.13/dist-packages/imageio/core/util.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/usr/local/lib/python3.13/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffm

In [5]:
transformer = transforms.Compose([
        transforms.ToTensor(),
        #transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

transformer_Arcface = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

detransformer = transforms.Compose([
        transforms.Normalize([0, 0, 0], [1/0.229, 1/0.224, 1/0.225]),
        transforms.Normalize([-0.485, -0.456, -0.406], [1, 1, 1])
    ])

In [6]:
!sed -i 's/map_location=torch.device("cpu"))/map_location=torch.device("cpu"), weights_only=False)/' /content/SimSwap/models/fs_model.py
!grep -n "torch.load" /content/SimSwap/models/fs_model.py

64:        netArc_checkpoint = torch.load(netArc_checkpoint, map_location=torch.device("cpu"), weights_only=False)


In [7]:
!wget -q https://github.com/deepinsight/insightface/releases/download/v0.7/antelopev2.zip -O /content/antelopev2.zip
!unzip -o -q /content/antelopev2.zip -d /content/antelopev2_tmp
!mkdir -p /content/SimSwap/insightface_func/models/antelope
!cp $(find /content/antelopev2_tmp -name scrfd_10g_bnkps.onnx) /content/SimSwap/insightface_func/models/antelope/
!ls -la /content/SimSwap/insightface_func/models/antelope

total 16536
drwxr-xr-x 2 root root     4096 Oct  4 06:38 .
drwxr-xr-x 3 root root     4096 Oct  4 06:38 ..
-rw-r--r-- 1 root root 16923827 Oct  4 06:38 scrfd_10g_bnkps.onnx


In [8]:
import numpy as np
np.float = float   # 옛 코드의 np.float를 Python 기본 float로 연결
np.int = int       # 같은 이유로 없어진 np.int도 미리 연결 (다음 오류 예방)

In [9]:
opt = TestOptions()
opt.initialize()
opt.parser.add_argument('-f') ## dummy arg to avoid bug
opt = opt.parse()
opt.pic_a_path = './demo_file/Iron_man.jpg' ## or replace it with image from your own google drive
opt.video_path = './demo_file/multi_people_1080p.mp4' ## or replace it with video from your own google drive
opt.output_path = './output/demo.mp4'
opt.temp_path = './tmp'
opt.Arc_path = './arcface_model/arcface_checkpoint.tar'
opt.isTrain = False
opt.use_mask = True  ## new feature up-to-date
opt.crop_size = 224  ## 받은 가중치(people)가 224용이라 맞춰 줌

crop_size = opt.crop_size

torch.nn.Module.dump_patches = True
model = create_model(opt)
model.eval()

app = Face_detect_crop(name='antelope', root='./insightface_func/models')
app.prepare(ctx_id= 0, det_thresh=0.6, det_size=(640,640))

with torch.no_grad():
    pic_a = opt.pic_a_path
    # img_a = Image.open(pic_a).convert('RGB')
    img_a_whole = cv2.imread(pic_a)
    img_a_align_crop, _ = app.get(img_a_whole,crop_size)
    img_a_align_crop_pil = Image.fromarray(cv2.cvtColor(img_a_align_crop[0],cv2.COLOR_BGR2RGB))
    img_a = transformer_Arcface(img_a_align_crop_pil)
    img_id = img_a.view(-1, img_a.shape[0], img_a.shape[1], img_a.shape[2])

    # convert numpy to tensor
    img_id = img_id.cuda()

    #create latent id
    img_id_downsample = F.interpolate(img_id, size=(112,112))
    latend_id = model.netArc(img_id_downsample)
    latend_id = latend_id.detach().to('cpu')
    latend_id = latend_id/np.linalg.norm(latend_id,axis=1,keepdims=True)
    latend_id = latend_id.to('cuda')

    #video_swap(opt.video_path, latend_id, model, app, opt.output_path, temp_results_dir=opt.temp_path, use_mask=opt.use_mask)

------------ Options -------------
Arc_path: arcface_model/arcface_checkpoint.tar
aspect_ratio: 1.0
batchSize: 8
checkpoints_dir: ./checkpoints
cluster_path: features_clustered_010.npy
crop_size: 512
data_type: 32
dataroot: ./datasets/cityscapes/
display_winsize: 512
engine: None
export_onnx: None
f: /root/.local/share/jupyter/runtime/kernel-2754dca5-17b9-4407-b28a-ce211e37d336.json
feat_num: 3
fineSize: 512
fp16: False
gpu_ids: [0]
how_many: 50
id_thres: 0.03
image_size: 224
input_nc: 3
instance_feat: False
isTrain: False
label_feat: False
label_nc: 0
latent_size: 512
loadSize: 1024
load_features: False
local_rank: 0
max_dataset_size: inf
multisepcific_dir: ./demo_file/multispecific
nThreads: 2
n_blocks_global: 6
n_blocks_local: 3
n_clusters: 10
n_downsample_E: 4
n_downsample_global: 3
n_local_enhancers: 1
name: people
nef: 16
netG: global
ngf: 64
niter_fix_global: 0
no_flip: False
no_instance: False
no_simswaplogo: False
norm: batch
norm_G: spectralspadesyncbatch3x3
ntest: inf
onnx: 

  latend_id = latend_id/np.linalg.norm(latend_id,axis=1,keepdims=True)



In [10]:
import os
from util.reverse2original import reverse2wholeimage
from util.norm import SpecificNorm
from parsing_model.model import BiSeNet

target_path = './demo_file/specific1.png'   # 다른 사람 사진 (4장 모두 같은 target)
os.makedirs('./output', exist_ok=True)

# 한 번만: 합성한 얼굴을 자연스럽게 붙이는 parsing 모델
net = BiSeNet(n_classes=19).cuda()
net.load_state_dict(torch.load('./parsing_model/checkpoint/79999_iter.pth'))
net.eval()
spNorm = SpecificNorm()

# 사진마다 반복
for name in ['person_a_1', 'person_a_2', 'person_a_3', 'person_a_4']:
    source_path = f'/content/{name}.jpg'
    output_path = f'./output/swap_{name}.jpg'

    with torch.no_grad():
        # source → 명세서
        img_a_whole = cv2.imread(source_path)
        img_a_align_crop, _ = app.get(img_a_whole, crop_size)
        img_a = transformer_Arcface(Image.fromarray(cv2.cvtColor(img_a_align_crop[0], cv2.COLOR_BGR2RGB)))
        img_id = img_a.unsqueeze(0).cuda()
        latend_id = model.netArc(F.interpolate(img_id, size=(112, 112)))
        latend_id = F.normalize(latend_id, p=2, dim=1)

        # target 얼굴을 명세서대로 다시 그리기
        img_b_whole = cv2.imread(target_path)
        img_b_align_crop_list, b_mat_list = app.get(img_b_whole, crop_size)
        swap_result_list, b_crop_tensor_list = [], []
        for b_crop in img_b_align_crop_list:
            b_tensor = torch.from_numpy(cv2.cvtColor(b_crop, cv2.COLOR_BGR2RGB)).permute(2, 0, 1).float().div(255)[None].cuda()
            swap_result_list.append(model(None, b_tensor, latend_id, None, True)[0])
            b_crop_tensor_list.append(b_tensor)

        # 원래 target 사진에 붙여서 저장
        reverse2wholeimage(b_crop_tensor_list, swap_result_list, b_mat_list, crop_size, img_b_whole, None,
                           output_path, no_simswaplogo=True, pasring_model=net, use_mask=True, norm=spNorm)

    print('저장 완료:', output_path)

Downloading: "https://download.pytorch.org/models/resnet18-5c106cde.pth" to /root/.cache/torch/hub/checkpoints/resnet18-5c106cde.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 263MB/s]
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]



저장 완료: ./output/swap_person_a_1.jpg
저장 완료: ./output/swap_person_a_2.jpg
저장 완료: ./output/swap_person_a_3.jpg
저장 완료: ./output/swap_person_a_4.jpg


In [11]:
def arc_embed(img_bgr):
    # SimSwap과 같은 방식: 얼굴 찾기·정렬 → 정규화 → 112×112 → ArcFace → 길이 1로 맞춤
    crops, _ = app.get(img_bgr, crop_size)
    x = transformer_Arcface(Image.fromarray(cv2.cvtColor(crops[0], cv2.COLOR_BGR2RGB))).unsqueeze(0).cuda()
    with torch.no_grad():
        e = model.netArc(F.interpolate(x, size=(112, 112)))
    return F.normalize(e, p=2, dim=1), crops[0]

# 한 번만: target 원본의 명세서 (4장 모두 같은 target)
tgt_emb, _ = arc_embed(cv2.imread(target_path))

# 사진마다 반복
for name in ['person_a_1', 'person_a_2', 'person_a_3', 'person_a_4']:
    src_emb, src_crop = arc_embed(cv2.imread(f'/content/{name}.jpg'))         # 내 사진
    res_emb, res_crop = arc_embed(cv2.imread(f'./output/swap_{name}.jpg'))    # 합성 결과

    # 정렬된 얼굴을 저장해서 눈으로 확인
    cv2.imwrite(f'./output/source_crop_{name}.jpg', src_crop)
    cv2.imwrite(f'./output/result_crop_{name}.jpg', res_crop)

    # 길이가 1인 벡터끼리 곱해서 더하면 코사인 유사도
    print(f'{name}: 결과 vs 내 사진 {(res_emb * src_emb).sum().item():.3f} | '
          f'결과 vs target {(res_emb * tgt_emb).sum().item():.3f} | '
          f'내 사진 vs target {(src_emb * tgt_emb).sum().item():.3f}')

person_a_1: 결과 vs 내 사진 0.729 | 결과 vs target 0.201 | 내 사진 vs target 0.062
person_a_2: 결과 vs 내 사진 0.749 | 결과 vs target 0.134 | 내 사진 vs target -0.049
person_a_3: 결과 vs 내 사진 0.713 | 결과 vs target 0.149 | 내 사진 vs target -0.027
person_a_4: 결과 vs 내 사진 0.745 | 결과 vs target 0.144 | 내 사진 vs target -0.010


In [20]:
# ===== Pilot: SimSwap 안의 ArcFace를 속이는 PGD =====
mean = torch.tensor([0.485, 0.456, 0.406], device='cuda').view(1, 3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225], device='cuda').view(1, 3, 1, 1)

def embed(x):
    # x: (1, 3, 224, 224), RGB, 0~1
    # SimSwap과 같은 순서: 정규화 → 112×112 → ArcFace → 길이 1로 맞춤
    x = x.contiguous()   # permute로 만든 텐서는 메모리가 순서대로 놓여 있지 않음 → ArcFace의 view()가 실패하므로 정리
    x = (x - mean) / std
    x = F.interpolate(x, size=(112, 112))
    return F.normalize(model.netArc(x), p=2, dim=1)

# [바뀐 부분 시작] ------------------------------------------------------------
SHIFTS = [(0, 0), (0, 1), (1, 0), (1, 1)]   # (세로, 가로) 이동. 2×2 중 어느 칸이 뽑혀도 통하도록 4가지 모두 계산

def shift(x, dy, dx):
    return torch.roll(x, shifts=(dy, dx), dims=(2, 3))

def pgd_identity(face, epsilon, steps=30):
    # 랜덤 출발 PGD + 4가지 이동의 평균 유사도를 낮춤 (모든 칸에 기울기가 생기도록)
    alpha = 2.5 * epsilon / steps
    orig_embs = [embed(shift(face, dy, dx)).detach() for dy, dx in SHIFTS]   # 이동마다 원본 명세서
    adv = (face + torch.empty_like(face).uniform_(-epsilon, epsilon)).clamp(0, 1).detach()
    for _ in range(steps):
        adv = adv.detach().requires_grad_(True)
        sims = [F.cosine_similarity(embed(shift(adv, dy, dx)), orig).mean()
                for (dy, dx), orig in zip(SHIFTS, orig_embs)]
        loss = sum(sims) / len(sims)         # 4가지 위치의 평균 유사도
        model.netArc.zero_grad()
        loss.backward()
        adv = adv - alpha * adv.grad.sign()  # 유사도를 낮추는 방향으로 이동
        perturbation = torch.clamp(adv - face, -epsilon, epsilon)
        adv = torch.clamp(face + perturbation, 0, 1)
    return adv.detach()
# [바뀐 부분 끝] --------------------------------------------------------------

def load_crop(path):
    # SimSwap이 정렬해서 잘라낸 224×224 얼굴 (BGR → RGB 텐서, 0~1)
    crops, _ = app.get(cv2.imread(path), crop_size)
    rgb = cv2.cvtColor(crops[0], cv2.COLOR_BGR2RGB)
    return torch.from_numpy(rgb).permute(2, 0, 1).float().div(255)[None].cuda()

def save_png(x, path):
    img = (x[0].permute(1, 2, 0).cpu().numpy() * 255).round().astype(np.uint8)
    cv2.imwrite(path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

os.makedirs('./protected', exist_ok=True)
names = ['person_a_1', 'person_a_2', 'person_a_3', 'person_a_4']
epsilons = [4, 8, 12]   # /255

for name in names:
    face = load_crop(f'/content/{name}.jpg')
    save_png(face, f'./protected/{name}_eps0.png')      # 노이즈 없는 crop (비교용)
    for e in epsilons:
        adv = pgd_identity(face, e / 255)
        save_png(adv, f'./protected/{name}_eps{e}.png')
        with torch.no_grad():
            sim = F.cosine_similarity(embed(adv), embed(face)).item()
            sim_shift = F.cosine_similarity(embed(shift(adv, 0, 1)), embed(shift(face, 0, 1))).item()
        print(f'{name} eps={e:>2}/255: 보호 vs 원본 (ArcFace) {sim:.3f} | 1픽셀 이동 후 {sim_shift:.3f}')

person_a_1 eps= 4/255: 보호 vs 원본 (ArcFace) -0.336 | 1픽셀 이동 후 -0.416
person_a_1 eps= 8/255: 보호 vs 원본 (ArcFace) -0.696 | 1픽셀 이동 후 -0.693
person_a_1 eps=12/255: 보호 vs 원본 (ArcFace) -0.723 | 1픽셀 이동 후 -0.732
person_a_2 eps= 4/255: 보호 vs 원본 (ArcFace) -0.198 | 1픽셀 이동 후 -0.274
person_a_2 eps= 8/255: 보호 vs 원본 (ArcFace) -0.642 | 1픽셀 이동 후 -0.744
person_a_2 eps=12/255: 보호 vs 원본 (ArcFace) -0.774 | 1픽셀 이동 후 -0.814
person_a_3 eps= 4/255: 보호 vs 원본 (ArcFace) -0.132 | 1픽셀 이동 후 0.040
person_a_3 eps= 8/255: 보호 vs 원본 (ArcFace) -0.474 | 1픽셀 이동 후 -0.621
person_a_3 eps=12/255: 보호 vs 원본 (ArcFace) -0.702 | 1픽셀 이동 후 -0.762
person_a_4 eps= 4/255: 보호 vs 원본 (ArcFace) -0.057 | 1픽셀 이동 후 -0.089
person_a_4 eps= 8/255: 보호 vs 원본 (ArcFace) -0.618 | 1픽셀 이동 후 -0.690
person_a_4 eps=12/255: 보호 vs 원본 (ArcFace) -0.735 | 1픽셀 이동 후 -0.799


In [15]:
face = load_crop('/content/person_a_1.jpg')
orig = embed(face).detach()
x = face.clone().requires_grad_(True)
loss = F.cosine_similarity(embed(x), orig).mean()
loss.backward()
print('유사도:', loss.item())
print('기울기 최대 크기:', x.grad.abs().max().item())
print('기울기가 0이 아닌 픽셀 비율:', (x.grad != 0).float().mean().item())

유사도: 1.0
기울기 최대 크기: 0.0
기울기가 0이 아닌 픽셀 비율: 0.0


In [21]:
def png_to_tensor(path):
    rgb = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    return torch.from_numpy(rgb).permute(2, 0, 1).float().div(255)[None].cuda()

def swap_with_latent(latend_id, output_path):
    # 명세서(latend_id)를 받아 target에 합성하고 저장 (4장 합성 셀과 같은 과정)
    img_b_whole = cv2.imread(target_path)
    crops_b, mats_b = app.get(img_b_whole, crop_size)
    tensors_b, results = [], []
    with torch.no_grad():
        for b_crop in crops_b:
            b = torch.from_numpy(cv2.cvtColor(b_crop, cv2.COLOR_BGR2RGB)).permute(2, 0, 1).float().div(255)[None].cuda()
            results.append(model(None, b, latend_id, None, True)[0])
            tensors_b.append(b)
        reverse2wholeimage(tensors_b, results, mats_b, crop_size, img_b_whole, None,
                           output_path, no_simswaplogo=True, pasring_model=net, use_mask=True, norm=spNorm)

def add_margin(img_bgr, pad=112):
    # 얼굴만 꽉 찬 224×224 사진 둘레에 검은 여백을 붙여 검출 모델이 얼굴을 찾기 쉽게 함
    return cv2.copyMakeBorder(img_bgr, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(0, 0, 0))

os.makedirs('./output_pilot', exist_ok=True)
for name in names:
    me_emb, _ = arc_embed(cv2.imread(f'/content/{name}.jpg'))   # 채점 기준: 원본 내 사진
    for e in [0] + epsilons:
        png = f'./protected/{name}_eps{e}.png'

        # (A) 직접: 보호 얼굴을 그대로 ArcFace에 넣음
        with torch.no_grad():
            lat_a = embed(png_to_tensor(png))
        swap_with_latent(lat_a, f'./output_pilot/{name}_eps{e}_direct.jpg')
        ra, _ = arc_embed(cv2.imread(f'./output_pilot/{name}_eps{e}_direct.jpg'))
        score_a = f'{(ra * me_emb).sum().item():.3f}'

        # (B) 재정렬: SimSwap이 보호 사진에서 얼굴을 다시 찾고 다시 정렬
        padded = add_margin(cv2.imread(png))
        if app.get(padded, crop_size) is None:
            score_b = '검출 실패'
        else:
            lat_b, _ = arc_embed(padded)
            swap_with_latent(lat_b, f'./output_pilot/{name}_eps{e}_realign.jpg')
            rb, _ = arc_embed(cv2.imread(f'./output_pilot/{name}_eps{e}_realign.jpg'))
            score_b = f'{(rb * me_emb).sum().item():.3f}'

        print(f'{name} eps={e:>2}/255 | 결과 vs 내 사진 (ArcFace)  (A) 직접 {score_a}  (B) 재정렬 {score_b}')

person_a_1 eps= 0/255 | 결과 vs 내 사진 (ArcFace)  (A) 직접 0.729  (B) 재정렬 0.774
person_a_1 eps= 4/255 | 결과 vs 내 사진 (ArcFace)  (A) 직접 -0.181  (B) 재정렬 0.005
person_a_1 eps= 8/255 | 결과 vs 내 사진 (ArcFace)  (A) 직접 -0.353  (B) 재정렬 0.010
person_a_1 eps=12/255 | 결과 vs 내 사진 (ArcFace)  (A) 직접 -0.443  (B) 재정렬 -0.139
person_a_2 eps= 0/255 | 결과 vs 내 사진 (ArcFace)  (A) 직접 0.749  (B) 재정렬 0.684
person_a_2 eps= 4/255 | 결과 vs 내 사진 (ArcFace)  (A) 직접 -0.127  (B) 재정렬 0.251
person_a_2 eps= 8/255 | 결과 vs 내 사진 (ArcFace)  (A) 직접 -0.455  (B) 재정렬 -0.077
person_a_2 eps=12/255 | 결과 vs 내 사진 (ArcFace)  (A) 직접 -0.544  (B) 재정렬 -0.153
person_a_3 eps= 0/255 | 결과 vs 내 사진 (ArcFace)  (A) 직접 0.713  (B) 재정렬 0.699
person_a_3 eps= 4/255 | 결과 vs 내 사진 (ArcFace)  (A) 직접 -0.178  (B) 재정렬 0.281
person_a_3 eps= 8/255 | 결과 vs 내 사진 (ArcFace)  (A) 직접 -0.357  (B) 재정렬 0.085
person_a_3 eps=12/255 | 결과 vs 내 사진 (ArcFace)  (A) 직접 -0.387  (B) 재정렬 -0.230
person_a_4 eps= 0/255 | 결과 vs 내 사진 (ArcFace)  (A) 직접 0.745  (B) 재정렬 0.733
person_a_4 eps= 4/255 | 결

In [23]:
face = load_crop('/content/person_a_1.jpg')
adv = png_to_tensor('./protected/person_a_1_eps8.png')
with torch.no_grad():
    for dx in [0, 1, 2]:
        # 보호 얼굴과 원본 얼굴을 똑같이 dx픽셀 밀어서 비교 (밀기 자체의 영향은 빼고 노이즈 효과만 봄)
        sim = F.cosine_similarity(embed(torch.roll(adv, dx, dims=3)),
                                  embed(torch.roll(face, dx, dims=3))).item()
        print(f'{dx}픽셀 이동: 보호 얼굴 vs 원본 얼굴 {sim:.3f}')

0픽셀 이동: 보호 얼굴 vs 원본 얼굴 -0.690
1픽셀 이동: 보호 얼굴 vs 원본 얼굴 -0.687
2픽셀 이동: 보호 얼굴 vs 원본 얼굴 -0.449
